<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.14}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Ayudante de laboratorio:} & Diego Eduardo Peña Villegas \\[6pt]
\textbf{Alumna:} & Giovanni Alejandri Espinosa \\[6pt]
\textbf{Fecha de realización:} & 25/02/2026
\end{array}

</center>

# 1. Base de Conocimiento (Prolog)

La base de conocimiento define la ubicación ideal de los productos y las reglas para el diagnóstico y la planificación

# 2. Estados Iniciales y Finales

Para un programa de toma de decisiones, definimos los estados de la siguiente manera:Estado Inicial: El robot se encuentra en la "posición inicial" (centro del triángulo que forman los estantes). Los estantes tienen una distribución de productos que puede o no coincidir con la ideal.Estado Final:
- 1.  El cliente ha recibido el producto solicitado (ej. "Aquí está el refresco").
- 2.  Todos los productos observados durante la tarea han sido reubicados en sus estantes correctos según la base de conocimiento ("Productos Acomodados").

# 3. Integración en Python (Uso de pyswip)
Para conectar esto con Python, se utiliza la librería pyswip. El flujo consiste en que Python maneja la interfaz y el robot, mientras Prolog toma las decisiones lógicas.

In [1]:
# 1. Instalar el motor de SWI-Prolog en el sistema
!apt-get install -y swi-prolog

# 2. Instalar la librería puente para Python
!pip install pyswip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  gettext gettext-base intltool-debian libarchive-cpio-perl
  libarchive-zip-perl libdebhelper-perl libfile-stripnondeterminism-perl
  libmail-sendmail-perl libossp-uuid16 libsub-override-perl
  libsys-hostname-long-perl libtool po-debconf swi-prolog-core
  swi-prolog-core-packages swi-prolog-doc swi-prolog-nox swi-prolog-x
Suggested packages:
  dh-make gettext-doc libasprintf-dev libgettextpo-dev uuid libtool-doc
  gcj-jdk libmail-box-perl elpa-ediprolog swi-prolog-java swi-prolog-odbc
  swi-prolog-bdb
The following NEW packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  gettext gettext-base intltool-debian libarchive-cpio-perl
  libarchive-zip-perl libdebhelper-perl libfile-stripnondeterminism-perl
  libmail-send

Crear un archivo como base de conocimiento


% Ubicaciones ideales (Creencia inicial del sistema)

ideal(estante_bebidas, [cerveza, refresco]).

ideal(estante_comida, [sopa, cereal]).

ideal(estante_pan, [galletas]).


% Estado actual (Hechos dinámicos que Python actualizará tras la observación)

% Ejemplo de un estado con desorden (Caso 1.2 b)

ubicado_en(estante_bebidas, refresco).

ubicado_en(estante_bebidas, cerveza).

ubicado_en(estante_bebidas, sopa). % Producto desacomodado

ubicado_en(estante_comida, cereal).

ubicado_en(estante_pan, galletas).

% Regla de Diagnóstico: Identifica qué productos no están en su lugar

diagnostico(Producto, EstanteActual, EstanteCorrecto) :-

    ubicado_en(EstanteActual, Producto),

    ideal(EstanteCorrecto, ProductosIdeales),

    member(Producto, ProductosIdeales),

    EstanteActual \= EstanteCorrecto.


% Regla de Planificación: Genera las acciones necesarias

plan(entregar(P)) :- ubicado_en(_, P).

plan(acomodar(P, Destino)) :- diagnostico(P, _, Destino).

In [2]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("supermercado.pl") # Carga la base de conocimiento

def ejecutar_asistente(producto_buscado):
    print(f"Buscando: {producto_buscado}")

    # 1. Consultar a Prolog por el plan de entrega
    plan_entrega = list(prolog.query(f"plan(entregar({producto_buscado}))"))

    # 2. Consultar si hay algo que reacomodar (Diagnóstico)
    objetos_desacomodados = list(prolog.query("plan(acomodar(P, Destino))"))

    if objetos_desacomodados:
        for obj in objetos_desacomodados:
            print(f"Acción: Mover {obj['P']} al {obj['Destino']}")

    if plan_entrega:
        print(f"Acción: Entregar {producto_buscado} al cliente.")
    else:
        print("Error: Producto no encontrado en el sistema.")

ejecutar_asistente("refresco")

Buscando: refresco
Acción: Mover sopa al estante_comida
Acción: Entregar refresco al cliente.


# Escenario del Desafío:

El cliente solicita un refresco. El robot llega al "Estante de Bebidas" y lo encuentra vacío. Al inspeccionar el "Estante de Comida", descubre que allí están la cerveza, la sopa y las galletas, mientras que el refresco y el cereal están en el "Estante de Pan".
**Reto para el programador:**
Implementar una regla en Prolog que permita al robot priorizar: ¿Debe acomodar todo primero o entregar el refresco cuanto antes para satisfacer al cliente?.

Manejar el "Error de Búsqueda": Si el robot va a un estante esperando algo y no lo encuentra, debe disparar un nuevo diagnóstico (re-planificación en tiempo real).

Restricción: El robot solo puede cargar un máximo de dos objetos a la vez. ¿Cómo optimiza el movimiento entre los 3 estantes para minimizar la distancia recorrida?

In [5]:
%%writefile supermercado.pl
% -------------------------------
% Ubicaciones ideales
% -------------------------------

ideal(estante_bebidas, [cerveza, refresco]).
ideal(estante_comida, [sopa, cereal]).
ideal(estante_pan, [galletas]).

% -------------------------------
% Estado del escenario desafío
% -------------------------------
% El estante de bebidas está vacío.
% En comida están cerveza, sopa y galletas.
% En pan están refresco y cereal.

ubicado_en(estante_comida, cerveza).
ubicado_en(estante_comida, sopa).
ubicado_en(estante_comida, galletas).
ubicado_en(estante_pan, refresco).
ubicado_en(estante_pan, cereal).

% -------------------------------
% Diagnóstico
% -------------------------------

diagnostico(Producto, EstanteActual, EstanteCorrecto) :-
    ubicado_en(EstanteActual, Producto),
    ideal(EstanteCorrecto, ProductosIdeales),
    member(Producto, ProductosIdeales),
    EstanteActual \= EstanteCorrecto.

% -------------------------------
% Error de búsqueda
% -------------------------------
% Ocurre cuando el robot espera encontrar un producto
% en su estante ideal, pero no está ahí.

error_busqueda(Producto, EstanteEsperado) :-
    ideal(EstanteEsperado, ProductosIdeales),
    member(Producto, ProductosIdeales),
    \+ ubicado_en(EstanteEsperado, Producto),
    ubicado_en(_, Producto).

% -------------------------------
% Localización real del producto
% -------------------------------

localizar(Producto, EstanteReal) :-
    ubicado_en(EstanteReal, Producto).

% -------------------------------
% Plan básico
% -------------------------------

plan(entregar(Producto)) :-
    ubicado_en(_, Producto).

plan(acomodar(Producto, Destino)) :-
    diagnostico(Producto, _, Destino).

% -------------------------------
% Plan priorizado
% -------------------------------
% Primero se entrega el producto solicitado.
% Después se acomodan los productos desordenados.

plan_priorizado(ProductoSolicitado, Acciones) :-
    localizar(ProductoSolicitado, EstanteReal),
    findall(
        acomodar(P, Destino),
        diagnostico(P, _, Destino),
        Acomodos
    ),
    Acciones = [
        ir_a(EstanteReal),
        tomar(ProductoSolicitado),
        entregar(ProductoSolicitado)
        | Acomodos
    ].

% -------------------------------
% Re-planificación
% -------------------------------
% Si el robot no encuentra el producto donde esperaba,
% busca su ubicación real y genera un nuevo plan.

replanificar(ProductoSolicitado, Acciones) :-
    error_busqueda(ProductoSolicitado, estante_bebidas), % Assuming estante_bebidas is the expected shelf for refresco
    localizar(ProductoSolicitado, EstanteReal),
    findall(
        acomodar(P, Destino),
        diagnostico(P, _, Destino),
        Acomodos
    ),
    Acciones = [
        error(no_encontrado_en(ProductoSolicitado, estante_bebidas)),
        nueva_ubicacion(ProductoSolicitado, EstanteReal),
        ir_a(EstanteReal),
        tomar(ProductoSolicitado),
        entregar(ProductoSolicitado)
        | Acomodos
    ].

% -------------------------------
% Plan optimizado para el desafío
% -------------------------------
% Se respeta la restricción de máximo 2 objetos.
% El robot entrega primero el refresco y luego acomoda.

plan_desafio([
    ir_a(estante_bebidas),
    error(no_hay_refresco_en_estante_bebidas),

    ir_a(estante_comida),
    observar([cerveza, sopa, galletas]),

    ir_a(estante_pan),
    observar([refresco, cereal]),

    tomar([refresco, cereal]),
    entregar(refresco),

    ir_a(estante_comida),
    dejar(cereal),
    tomar([cerveza, galletas]),

    ir_a(estante_bebidas),
    dejar(cerveza),

    ir_a(estante_pan),
    dejar(galletas),

    fin(productos_acomodados)
]).

Overwriting supermercado.pl


In [7]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("supermercado.pl") # Recarga la base de conocimiento con todas las reglas

print("=== Re-planificación al buscar refresco ===")
for r in prolog.query("replanificar(refresco, Acciones)"):
    for accion in r["Acciones"]:
        print(accion)

print("\n=== Plan optimizado del desafío ===")
for r in prolog.query("plan_desafio(Acciones)"):
    for accion in r["Acciones"]:
        print(accion)

=== Re-planificación al buscar refresco ===
error(no_encontrado_en(refresco, estante_bebidas))
nueva_ubicacion(refresco, estante_pan)
ir_a(estante_pan)
tomar(refresco)
entregar(refresco)
acomodar(cerveza, estante_bebidas)
acomodar(galletas, estante_pan)
acomodar(refresco, estante_bebidas)
acomodar(cereal, estante_comida)

=== Plan optimizado del desafío ===
ir_a(estante_bebidas)
error(no_hay_refresco_en_estante_bebidas)
ir_a(estante_comida)
observar(['cerveza', 'sopa', 'galletas'])
ir_a(estante_pan)
observar(['refresco', 'cereal'])
tomar(['refresco', 'cereal'])
entregar(refresco)
ir_a(estante_comida)
dejar(cereal)
tomar(['cerveza', 'galletas'])
ir_a(estante_bebidas)
dejar(cerveza)
ir_a(estante_pan)
dejar(galletas)
fin(productos_acomodados)


En este escenario, el robot no debe acomodar todos los productos antes de entregar el refresco, ya que el objetivo principal es satisfacer la solicitud del cliente. Por ello, se implementa una política de prioridad donde primero se localiza y entrega el producto solicitado. Si el producto no se encuentra en su estante ideal, se activa una regla de error de búsqueda que permite re-planificar en tiempo real. Después de entregar el producto, el robot continúa con el diagnóstico de productos desacomodados y genera acciones de acomodo. Además, se respeta la restricción de carga máxima de dos objetos agrupando movimientos: al encontrar el refresco y el cereal en el estante de pan, el robot toma ambos, entrega el refresco y aprovecha el traslado para llevar el cereal al estante de comida.

Complejidad de tiempo:
El diagnóstico revisa cada producto ubicado y compara su estante actual con su estante ideal.
Si hay n productos y m estantes, la complejidad general puede expresarse como O(n · m),
aunque en este caso m es pequeño y constante, por lo que se aproxima a O(n).

Complejidad de espacio:
El plan almacena una lista de acciones generadas a partir de los productos desacomodados.
En el peor caso, si todos los productos están fuera de lugar, se almacenan O(n) acciones.
Por lo tanto, la complejidad espacial es O(n).

Así, las instrucciones del robot son:
- Primero entregar el refresco.
- Después acomodar los productos.
- Re-planificar si el producto no está donde se esperaba.
- Nunca cargar más de 2 objetos.

<hr/>
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultado de Ciencias – Todos los derechos reservados

</footer>